# Lesson 05 Lab — Explicit CUDA Control and Error Boundaries

**Puzzle:** When thread blocks, synchronization, nvcc, and reviewable source change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates thread blocks, synchronization, nvcc, and reviewable source and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

CUDA C++ makes the scalar thread index, block geometry, launch, synchronization, and error checks explicit. Triton moves many of those choices into a blocked program and compiler. A fair comparison needs both implementations and a toolchain that can actually build the CUDA path.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["thread blocks, synchronization, nvcc, and reviewable source"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: reviewed CUDA C++ source plus toolchain availability. Candidate: reviewed Triton kernel or explicit model described below.

Source code availability is not native execution evidence. Without nvcc, a CUDA latency cell must remain unmeasured.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 5
LESSON_TITLE = 'Explicit CUDA Control and Error Boundaries'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260818
}


## 5. Freeze the experiment

**Experiment:** Run the Triton implementation, inspect nvcc availability, and retain an equivalent CUDA C++ source file with launch checks.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": false,
  "secondary": 0.021167999133467674,
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "cuda_source": "vector_affine.cu",
    "cuda_compiled": false,
    "reason": "nvcc unavailable",
    "triton_samples_ms": [
      0.03328000009059906,
      0.026048000901937485,
      0.02284800074994564,
      0.022016000002622604,
      0.02067199908196926,
      0.023231999948620796,
      0.022655999287962914,
      0.022624000906944275,
      0.02175999991595745,
      0.021663999184966087,
      0.020031999796628952,
      0.022016000002622604,
      0.020096000283956528,
      0.020416000857949257,
      0.02035200037062168,
      0.019231999292969704,
      0.01894400082528591,
      0.01942400075495243,
      0.019872000440955162,
      0.020479999482631683
    ]
  }
}
The Triton path ran correctly in 0.0212 ms. nvcc availability was False, so the explicit CUDA source is reviewable but is not presented as a native measurement.


## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| nvcc available | false |
| Triton median | 0.0212 ms |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The Triton path ran correctly in 0.0212 ms. nvcc availability was False, so the explicit CUDA source is reviewable but is not presented as a native measurement.

The installed toolchain or API surface was inspected. An available symbol or source file is not reported as native performance on an unexecuted backend.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'compatibility-probe',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Use the explicit CUDA result only after the recorded Toolkit builds and runs it; until then it is a review artifact.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 5,
  "title": "Explicit CUDA Control and Error Boundaries",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260818
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "primary": false,
    "secondary": 0.021167999133467674,
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "cuda_source": "vector_affine.cu",
      "cuda_compiled": false,
      "reason": "nvcc unavailable",
      "triton_samples_ms": [
        0.03328000009059906,
        0.026048000901937485,
        0.02284800074994564,
        0.022016000002622604,
        0.02067199908196926,
        0.023231999948620796,
        0.022655999287962914,
        0.022624000906944275,
        0.02175999991595745,
        0.021663999184966087,


## 10. Make the bounded decision

> Use the explicit CUDA result only after the recorded Toolkit builds and runs it; until then it is a review artifact.

**Failure analysis:** Source code availability is not native execution evidence. Without nvcc, a CUDA latency cell must remain unmeasured.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
